# Gemma 3 12B Instruct — PersianQA QLoRA on a single Colab T4

**Standalone, text-only research notebook.** No GitHub clone, local package, or uploaded dataset files are required. This is a memory-constrained adaptation of the completed Gemma 3 4B experiment.

**Objective:** Measure the change in PersianQA Exact Match, token F1 and abstention behavior between the **same** original 12B 4-bit checkpoint and its LoRA-tuned version.

**Important limitations**

- A **16 GB T4 is not guaranteed** to train Gemma 3 12B. The configuration deliberately uses the *standard* 4-bit BnB checkpoint (`unsloth/gemma-3-12b-it-bnb-4bit`), not the larger dynamic quantization checkpoint.
- Training parameters: batch 1, gradient accumulation 8, rank 8, 512-token maximum sequence, 250 optimizer steps. These are feasibility-first settings, **not** a tuned model configuration.
- Evaluations use the **same** test records, prompts, truncation budget, generation settings, and scoring for base and tuned models.
- The context limit is smaller than in the 4B experiment. The 4B and 12B scores **must not** be presented as a clean scaling comparison without rerunning them with matched settings.
- The official held-out PersianQA split is named `validation` by the Hugging Face data loader. It is never used for training or model selection.
- A 32-question smoke test is available but **cannot be reported as a research result**.

Use Colab: **Runtime → Change runtime type → T4 GPU**. Run cells in order. After installing dependencies, restart the runtime only if Colab requires it.

## 0. Dependencies and GPU checks

Use Unsloth for FP16 Gemma 3 training on a T4. Do not install or upgrade a second, incompatible PyTorch build after loading Unsloth.

In [ ]:
import os
import sys
import subprocess

assert os.path.isdir('/content'), 'This notebook is designed for Google Colab.'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth', 'datasets>=3,<5', 'huggingface_hub>=0.34',
    'sentencepiece', 'protobuf',
])
print('Installation complete. Restart Colab if prompted, then rerun the setup cells.')

Installation complete. Restart Colab if prompted, then rerun the setup cells.


In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable the T4 GPU in Colab runtime settings.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / (2 ** 30)
assert GPU_GIB >= 14.0, f'Insufficient VRAM: {GPU_GIB:.1f} GiB.'
print(f'GPU: {GPU_NAME}, {GPU_GIB:.1f} GiB | PyTorch: {torch.__version__}')

GPU: Tesla T4, 14.6 GiB | PyTorch: 2.11.0+cu130


## 1. Configuration

`RUN_STAGES` controls execution. The default runs everything in order. With Google Drive enabled, individual stages can be resumed on a later Colab session without discarding completed predictions or the saved adapter. Examples: `('baseline',)`, `('train',)`, `('finetuned', 'compare')`.

Set `SMOKE_TEST=True` initially to test compatibility. For final research results set it to `False`, rerun from the beginning, and retain the complete output folder.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import time
import unicodedata
import uuid

import numpy as np
from datasets import Dataset, load_dataset

SEED = 3407
MODEL_ID = 'unsloth/gemma-3-12b-it-bnb-4bit'  # Standard BnB, smaller than dynamic Unsloth 4-bit
DATASET_ID = 'SajjadAyoubi/persian_qa'
DATASET_REVISION = 'refs/convert/parquet'        # Avoid unsupported legacy persian_qa.py

SMOKE_TEST = False
RUN_STAGES = ('baseline', 'train', 'finetuned', 'compare')

MAX_STEPS = 250
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LORA_R = 8
LORA_ALPHA = 16
LEARNING_RATE = 1e-4
MAX_SEQ_LENGTH = 512
MAX_CONTEXT_TOKENS = 288
MAX_NEW_TOKENS = 72
VALIDATION_FRACTION = 0.10
NO_ANSWER_TEXT = 'پاسخی در متن وجود ندارد'
BOOTSTRAP_REPEATS = 2000
SAVE_EVERY_STEPS = 50

USE_GOOGLE_DRIVE = True
VALID_STAGES = ('baseline', 'train', 'finetuned', 'compare')
assert all(stage in VALID_STAGES for stage in RUN_STAGES)
assert 0 < MAX_CONTEXT_TOKENS < MAX_SEQ_LENGTH - MAX_NEW_TOKENS
assert LORA_R > 0 and LORA_ALPHA > 0

if SMOKE_TEST:
    MAX_STEPS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

OUTPUT_ROOT = Path('/content/gemma3_12b_persianqa')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/Gemma3_12B_PersianQA_T4')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Stages:', RUN_STAGES)
print('SMOKE TEST:', SMOKE_TEST, '(scores not reportable)' if SMOKE_TEST else '')
print('Output root:', OUTPUT_ROOT)

Mounted at /content/drive
Stages: ('baseline', 'train', 'finetuned', 'compare')
SMOKE TEST: False 
Output root: /content/drive/MyDrive/Gemma3_12B_PersianQA_T4


## 2. Load PersianQA directly from Hugging Face

The dataset repository exposes auto-converted Parquet at `refs/convert/parquet`. This loads the Hugging Face dataset **directly**, without legacy loading scripts or manually composed URLs. Keep its official held-out `validation` split untouched.

An internal validation set is formed from the *training split* by stable passage hashing to avoid passage overlap.

In [ ]:
raw = load_dataset(DATASET_ID, revision=DATASET_REVISION)
assert 'train' in raw, f'Expected training split. Found: {list(raw)}'
OFFICIAL_TEST_SPLIT = 'test' if 'test' in raw else 'validation'
assert OFFICIAL_TEST_SPLIT in raw, f'No held-out split found in {list(raw)}'

train_source = raw['train']
official_test = raw[OFFICIAL_TEST_SPLIT]
for subset in (train_source, official_test):
    assert {'question', 'context', 'answers'}.issubset(subset.column_names), subset.column_names
assert len(train_source) > 1000 and len(official_test) > 100

# Group questions by their original passage before splitting.
def passage_key(row):
    passage = str(row.get('title', '')) + '\n' + str(row['context'])
    return hashlib.sha256(passage.encode('utf-8')).hexdigest()

def to_internal_validation(row):
    return int(passage_key(row)[:8], 16) / 2**32 < VALIDATION_FRACTION

train_indices = [i for i, row in enumerate(train_source) if not to_internal_validation(row)]
val_indices = [i for i, row in enumerate(train_source) if to_internal_validation(row)]
train_data = train_source.select(train_indices).shuffle(seed=SEED)
internal_val = train_source.select(val_indices)
assert len(train_data) > 0 and len(internal_val) > 0
assert {passage_key(r) for r in train_data}.isdisjoint({passage_key(r) for r in internal_val})
assert {str(r['id']) for r in train_data}.isdisjoint({str(r['id']) for r in internal_val})

full_test_count = len(official_test)
if SMOKE_TEST:
    train_data = train_data.select(range(min(96, len(train_data))))
    official_test = official_test.select(range(min(32, len(official_test))))

print(f'Train: {len(train_data)} | internal dev: {len(internal_val)} | held-out: {len(official_test)} / {full_test_count}')
print('Columns:', train_data.column_names)
print('Example answers:', train_data[0]['answers'])

persian_qa/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.62MB            

persian_qa/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/165k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train: 8198 | internal dev: 810 | held-out: 930 / 930
Columns: ['id', 'title', 'context', 'question', 'answers']
Example answers: {'text': ['هر نوع بازی است که از کارت\u200cهای بازی به عنوان دستگاه اصلی که بازی با آن انجام می\u200cشود'], 'answer_start': [35]}


## 3. Experiment identity and metadata

Keep 12B results separate from 4B results and from small smoke tests. A compact configuration hash determines the output directory. Model and converted-dataset revisions are recorded in the metadata.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
MODEL_SHA = api.model_info(MODEL_ID).sha
PARQUET_SHA = api.dataset_info(DATASET_ID, revision=DATASET_REVISION).sha

EXPERIMENT_CONFIG = {
    'model_id': MODEL_ID,
    'model_sha': MODEL_SHA,
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'parquet_sha': PARQUET_SHA,
    'official_test_split': OFFICIAL_TEST_SPLIT,
    'seed': SEED,
    'smoke_test': SMOKE_TEST,
    'eval_rows': len(official_test),
    'max_steps': MAX_STEPS,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUMULATION,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'learning_rate': LEARNING_RATE,
    'max_seq_length': MAX_SEQ_LENGTH,
    'max_context_tokens': MAX_CONTEXT_TOKENS,
    'max_new_tokens': MAX_NEW_TOKENS,
    'validation_fraction': VALIDATION_FRACTION,
    'no_answer_text': NO_ANSWER_TEXT,
}
RUN_KEY = hashlib.sha256(
    json.dumps(EXPERIMENT_CONFIG, sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()[:12]
BASE_OUTPUT = OUTPUT_ROOT / (('smoke_' if SMOKE_TEST else 'full_') + RUN_KEY)
BASE_OUTPUT.mkdir(parents=True, exist_ok=True)

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

CONFIG_PATH = BASE_OUTPUT / 'experiment_config.json'
if CONFIG_PATH.exists():
    assert read_json(CONFIG_PATH) == EXPERIMENT_CONFIG, 'Configuration mismatch: use a fresh experiment directory.'
else:
    write_json(CONFIG_PATH, EXPERIMENT_CONFIG)
print('Experiment directory:', BASE_OUTPUT)
print('Model revision:', MODEL_SHA)
print('Converted dataset revision:', PARQUET_SHA)

Experiment directory: /content/drive/MyDrive/Gemma3_12B_PersianQA_T4/full_6f853880f866
Model revision: 826e729dbaeea4ecb143738eed2bcf3539ebf7bf
Converted dataset revision: 983eff5900bf5e494161529938e3de0ffc75e832


## 4. Load 12B checkpoint and tokenizer

`FastModel` applies Unsloth's Gemma 3 FP16/T4 compatibility patches. LoRA is added **only after** baseline inference. Skip loading the model when running only `compare` from previously saved files.

In [ ]:
model = None
tokenizer = None
requires_gpu_model = any(stage in RUN_STAGES for stage in ('baseline', 'train', 'finetuned'))
if requires_gpu_model:
    from unsloth import FastModel  # Must precede importing Trainer/Transformers model classes
    from transformers import AutoTokenizer

    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)

    model, _processor = FastModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    assert tokenizer.chat_template, 'The Gemma 3 tokenizer must provide the native chat template.'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    assert tokenizer.pad_token_id is not None

    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f'Model class: {type(model).__name__}')
    print(f'Free VRAM after load: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB')
    print('Tokenizer: PAD', tokenizer.pad_token_id, '| EOS', tokenizer.eos_token_id)
else:
    print('Compare-only mode: model is not loaded.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.2: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/397k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Model class: Gemma3ForConditionalGeneration
Free VRAM after load: 7.09 / 14.56 GiB
Tokenizer: PAD 0 | EOS 106


## 5. Native Gemma 3 prompts and answer-only labels

The same prompt builder is used for baseline, training, and tuned evaluation. The native tokenizer chat template creates Gemma 3 role delimiters. Label `-100` excludes the user question and context from the training loss.

A shortened context may remove a gold answer. Every truncated example is recorded so this limitation can be audited.

In [ ]:
def answer_texts(row):
    answers = row['answers']
    texts = answers.get('text', []) if isinstance(answers, dict) else []
    return [str(s).strip() for s in texts if s is not None and str(s).strip()]

def selected_training_answer(row):
    refs = answer_texts(row)
    return refs[0] if refs else NO_ANSWER_TEXT

def build_prompt(row):
    if tokenizer is None:
        raise RuntimeError('Load the model/tokenizer before calling build_prompt.')
    context_ids = tokenizer.encode(str(row['context']), add_special_tokens=False)
    selected_ids = context_ids[:MAX_CONTEXT_TOKENS]
    shortened = len(selected_ids) < len(context_ids)
    question = str(row['question']).strip()

    def render(text):
        user_text = (
            'با توجه به متن زیر، فقط پاسخ کوتاه و دقیق پرسش را بنویس. '
            f'اگر پاسخ در متن وجود ندارد، دقیقا بنویس: {NO_ANSWER_TEXT}. '
            'از توضیح اضافی خودداری کن.\n\n'
            f'متن: {text}\n\nسؤال: {question}'
        )
        return tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_text}],
            tokenize=False,
            add_generation_prompt=True,
        )

    while True:
        context = tokenizer.decode(selected_ids, skip_special_tokens=True)
        prompt = render(context)
        prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)
        if len(prompt_tokens) <= MAX_SEQ_LENGTH - MAX_NEW_TOKENS:
            break
        if not selected_ids:
            raise ValueError('The question and instructions exceed the configured sequence budget.')
        selected_ids = selected_ids[:-min(16, len(selected_ids))]
        shortened = True
    return prompt, context, shortened

def training_item(row):
    prompt, _, _ = build_prompt(row)
    answer = selected_training_answer(row)
    example = [{'role':'user', 'content':'USER_CONTENT'},
               {'role':'assistant', 'content':answer}]
    complete_template = tokenizer.apply_chat_template(example, tokenize=False, add_generation_prompt=False)
    placeholder_prefix = tokenizer.apply_chat_template(
        [{'role':'user','content':'USER_CONTENT'}], tokenize=False, add_generation_prompt=True
    )
    assert complete_template.startswith(placeholder_prefix), 'Unexpected Gemma chat template.'
    suffix = complete_template[len(placeholder_prefix):]

    prefix_ids = tokenizer.encode(prompt, add_special_tokens=False)
    target_ids = tokenizer.encode(suffix, add_special_tokens=False)
    if not target_ids or len(target_ids) > MAX_NEW_TOKENS:
        return None
    input_ids = prefix_ids + target_ids
    if len(input_ids) > MAX_SEQ_LENGTH:
        return None
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels': [-100] * len(prefix_ids) + target_ids,
    }

if tokenizer is not None:
    prompt_example, context_example, shortened = build_prompt(train_data[0])
    tokenized_example = training_item(train_data[0])
    assert tokenized_example is not None
    assert any(token != -100 for token in tokenized_example['labels'])
    assert tokenized_example['labels'][0] == -100
    print('Prompt characters:', len(prompt_example), '| truncated:', shortened)
    print(prompt_example[:400])

Prompt characters: 1272 | truncated: False
<bos><start_of_turn>user
با توجه به متن زیر، فقط پاسخ کوتاه و دقیق پرسش را بنویس. اگر پاسخ در متن وجود ندارد، دقیقا بنویس: پاسخی در متن وجود ندارد. از توضیح اضافی خودداری کن.

متن: کارت‌بازی یا ورق‌بازی یا پاسوربازی هر نوع بازی است که از کارت‌های بازی به عنوان دستگاه اصلی که بازی با آن انجام می‌شود، است. تعداد بیشماری بازی با ورق وجود دارد. برگزاری مسابقات بین‌المللی به‌طور رسمی قوانین استانداردی 


## 6. PersianQA metrics

EM and F1 operate on normalized Persian Unicode and use multiset token overlap. The abstention label is mapped to an empty answer for metric compatibility. The exact-phrase abstention rule is kept from the previous 4B experiment for consistency; it does **not** recognize every semantically equivalent refusal.

In [ ]:
PERSIAN_CHAR_MAP = str.maketrans({
    'ي':'ی','ى':'ی','ك':'ک','ۀ':'ه','ة':'ه','أ':'ا','إ':'ا','ٱ':'ا',
    '۰':'0','۱':'1','۲':'2','۳':'3','۴':'4','۵':'5','۶':'6','۷':'7','۸':'8','۹':'9',
    '٠':'0','١':'1','٢':'2','٣':'3','٤':'4','٥':'5','٦':'6','٧':'7','٨':'8','٩':'9',
})

def normalize_fa(text):
    text = unicodedata.normalize('NFKC', str(text)).translate(PERSIAN_CHAR_MAP)
    text = ''.join(
        ' ' if unicodedata.category(ch).startswith(('P','C'))
        else '' if unicodedata.category(ch).startswith('M') else ch
        for ch in text
    )
    return ' '.join(text.casefold().split())

NORMALIZED_ABSTENTION = normalize_fa(NO_ANSWER_TEXT)

def canonical_prediction(prediction):
    prediction = str(prediction).strip()
    return '' if normalize_fa(prediction) == NORMALIZED_ABSTENTION else prediction

def token_f1(prediction, reference):
    p_tokens = normalize_fa(prediction).split()
    r_tokens = normalize_fa(reference).split()
    if not p_tokens and not r_tokens: return 1.0
    if not p_tokens or not r_tokens: return 0.0
    overlap = sum((Counter(p_tokens) & Counter(r_tokens)).values())
    return 2 * overlap / (len(p_tokens) + len(r_tokens))

def score_example(prediction, references):
    pred = canonical_prediction(prediction)
    targets = references if references else ['']
    return {
        'em': max(float(normalize_fa(pred) == normalize_fa(ref)) for ref in targets),
        'f1': max(token_f1(pred, ref) for ref in targets),
        'gold_no_answer': not bool(references),
        'pred_no_answer': not bool(normalize_fa(pred)),
    }

def summarize_scores(records):
    if not records:
        raise ValueError('Cannot score an empty prediction set.')
    em = np.array([r['em'] for r in records], dtype=np.float64)
    f1 = np.array([r['f1'] for r in records], dtype=np.float64)
    gold_na = np.array([r['gold_no_answer'] for r in records], dtype=bool)
    pred_na = np.array([r['pred_no_answer'] for r in records], dtype=bool)
    tp = int((gold_na & pred_na).sum())
    fp = int((~gold_na & pred_na).sum())
    fn = int((gold_na & ~pred_na).sum())
    tn = int((~gold_na & ~pred_na).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    na_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        'n': len(records),
        'em': float(em.mean()), 'f1': float(f1.mean()),
        'answerable_f1': float(f1[~gold_na].mean()) if (~gold_na).any() else None,
        'unanswerable_em': float(em[gold_na].mean()) if gold_na.any() else None,
        'no_answer_precision': precision, 'no_answer_recall': recall,
        'no_answer_f1': na_f1,
        'no_answer_tp': tp, 'no_answer_fp': fp,
        'no_answer_fn': fn, 'no_answer_tn': tn,
    }

# CPU-side correctness checks.
assert normalize_fa('كِتاب، ۱۲۳') == normalize_fa('کتاب 123')
assert token_f1('الف الف', 'الف') == 2/3
assert token_f1('', '') == 1.0
assert token_f1('پاسخ', '') == 0.0
assert score_example(NO_ANSWER_TEXT, [])['em'] == 1.0
assert score_example('تهران', ['شیراز', 'تهران'])['em'] == 1.0
print('Metric unit tests: passed.')

Metric unit tests: passed.


## 7. Resumable prediction and evaluation

Each generated answer is appended to a JSONL file. A separate evaluation metadata file prevents mixing checkpoints or settings, and the loop resumes from an already completed prefix of examples. For a valid full benchmark both files must contain exactly `len(official_test)` records.

Interrupted evaluation writes partial predictions but **not** a completed metrics report.

In [ ]:
def evaluation_identity(stage):
    tag = 'untuned'
    if stage == 'finetuned':
        manifest_path = BASE_OUTPUT / 'training_manifest.json'
        assert manifest_path.exists(), 'Run fine-tuning first; adapter manifest is missing.'
        tag = read_json(manifest_path)['training_id']
    description = {
        'stage': stage, 'checkpoint': MODEL_ID, 'model_sha': MODEL_SHA,
        'adapter_tag': tag, 'parquet_sha': PARQUET_SHA,
        'eval_rows': len(official_test),
        'max_seq_length': MAX_SEQ_LENGTH,
        'max_context_tokens': MAX_CONTEXT_TOKENS,
        'max_new_tokens': MAX_NEW_TOKENS,
        'no_answer_text': NO_ANSWER_TEXT,
    }
    return description

def read_saved_records(name):
    file = BASE_OUTPUT / f'{name}.predictions.jsonl'
    if not file.exists(): return []
    records = []
    with file.open(encoding='utf-8') as reader:
        for line_number, line in enumerate(reader, 1):
            if not line.strip(): continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise RuntimeError(f'Corrupt {name} predictions at line {line_number}: {exc}')
    return records

def validate_saved_records(records, eval_ds):
    if len(records) > len(eval_ds):
        raise ValueError('Saved predictions contain more examples than the selected held-out split.')
    for index, saved in enumerate(records):
        row = eval_ds[index]
        expected_id = str(row.get('id', index))
        if saved['id'] != expected_id or saved['references'] != answer_texts(row):
            raise ValueError(f'Saved prediction mismatch at position {index}.')

def evaluate_model(active_model, name, eval_ds):
    if active_model is None:
        raise RuntimeError('No model loaded for evaluation.')
    active_model.eval()
    identity = evaluation_identity(name)
    manifest_path = BASE_OUTPUT / f'{name}.evaluation_manifest.json'
    if manifest_path.exists():
        assert read_json(manifest_path) == identity, f'{name}: evaluation identity changed.'
    else:
        assert not (BASE_OUTPUT / f'{name}.predictions.jsonl').exists(), 'Predictions without manifest: unsafe to resume.'
        write_json(manifest_path, identity)

    records = read_saved_records(name)
    validate_saved_records(records, eval_ds)
    print(f'{name}: resuming at {len(records)}/{len(eval_ds)}')
    pred_path = BASE_OUTPUT / f'{name}.predictions.jsonl'
    start = time.time()
    count_new = 0
    with pred_path.open('a', encoding='utf-8') as writer:
        for i in range(len(records), len(eval_ds)):
            row = eval_ds[i]
            prompt, truncated_context, was_truncated = build_prompt(row)
            ids = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
            ids = {key: value.to('cuda') for key, value in ids.items()}
            with torch.inference_mode():
                generated = active_model.generate(
                    **ids, do_sample=False, max_new_tokens=MAX_NEW_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                )
            output_ids = generated[0][ids['input_ids'].shape[-1]:]
            prediction = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
            references = answer_texts(row)
            record = {
                'id': str(row.get('id', i)),
                'question': str(row['question']),
                'references': references,
                'prediction': prediction,
                'context_was_truncated': was_truncated,
                'answer_present_in_truncated_context': (
                    any(normalize_fa(ref) in normalize_fa(truncated_context) for ref in references)
                    if references else None
                ),
                **score_example(prediction, references),
            }
            writer.write(json.dumps(record, ensure_ascii=False) + '\n')
            count_new += 1
            if (i + 1) % 10 == 0 or i + 1 == len(eval_ds):
                writer.flush()
            if (i + 1) % 25 == 0 or i + 1 == len(eval_ds):
                print(f'{name}: {i+1}/{len(eval_ds)}')
            del ids, generated, output_ids

    records = read_saved_records(name)
    validate_saved_records(records, eval_ds)
    assert len(records) == len(eval_ds), f'{name}: incomplete evaluation.'
    scores = summarize_scores(records)
    scores.update({
        'smoke_test': SMOKE_TEST,
        'model_id': MODEL_ID, 'model_sha': MODEL_SHA,
        'dataset_id': DATASET_ID, 'parquet_sha': PARQUET_SHA,
        'split': OFFICIAL_TEST_SPLIT,
        'truncated_contexts': sum(int(row['context_was_truncated']) for row in records),
        'answerable_gold_missing_after_truncation': sum(
            int(not row['answer_present_in_truncated_context'])
            for row in records if row['context_was_truncated'] and row['references']
        ),
        'elapsed_seconds_this_session': round(time.time() - start, 2),
        'new_predictions_this_session': count_new,
        'generation': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
    })
    write_json(BASE_OUTPUT / f'{name}.metrics.json', scores)
    return records, scores

print('Resumable evaluation ready.')

Resumable evaluation ready.


## 8. Baseline evaluation (unmodified Gemma 3 12B)

No adapter is attached during this phase. Predictions and metrics are saved to Drive. Re-running this cell reuses completed predictions that match the evaluation identity.

In [ ]:
baseline_rows = None
baseline_scores = None
if 'baseline' in RUN_STAGES:
    baseline_rows, baseline_scores = evaluate_model(model, 'baseline', official_test)
    print(json.dumps(baseline_scores, ensure_ascii=False, indent=2))
else:
    print('Baseline stage skipped.')

baseline: resuming at 0/930
baseline: 25/930
baseline: 50/930
baseline: 75/930
baseline: 100/930
baseline: 125/930
baseline: 150/930
baseline: 175/930
baseline: 200/930
baseline: 225/930
baseline: 250/930
baseline: 275/930
baseline: 300/930
baseline: 325/930
baseline: 350/930
baseline: 375/930
baseline: 400/930
baseline: 425/930
baseline: 450/930
baseline: 475/930
baseline: 500/930
baseline: 525/930
baseline: 550/930
baseline: 575/930
baseline: 600/930
baseline: 625/930
baseline: 650/930
baseline: 675/930
baseline: 700/930
baseline: 725/930
baseline: 750/930
baseline: 775/930
baseline: 800/930
baseline: 825/930
baseline: 850/930
baseline: 875/930
baseline: 900/930
baseline: 925/930
baseline: 930/930
{
  "n": 930,
  "em": 0.6043010752688172,
  "f1": 0.7534067731399157,
  "answerable_f1": 0.8044059892782206,
  "unanswerable_em": 0.6344086021505376,
  "no_answer_precision": 0.8894472361809045,
  "no_answer_recall": 0.6344086021505376,
  "no_answer_f1": 0.7405857740585773,
  "no_answer_tp"

## 9. Tokenize the training set and construct a memory-efficient collator

Only the assistant's answer tokens contribute to the loss. No validation or official test examples enter the training dataset.

In [ ]:
train_token_ds = None
skipped_long_targets = 0
if 'train' in RUN_STAGES:
    processed = []
    for example in train_data:
        item = training_item(example)
        if item is None:
            skipped_long_targets += 1
        else:
            processed.append(item)
    assert processed, 'No training examples fit in the current sequence budget.'
    train_token_ds = Dataset.from_list(processed)
    print('Tokenized training examples:', len(train_token_ds))
    print('Skipped due to length:', skipped_long_targets)
else:
    print('Training dataset stage skipped.')

def collate(batch):
    if tokenizer is None:
        raise RuntimeError('Tokenizer not available.')
    longest = max(len(item['input_ids']) for item in batch)
    width = min(MAX_SEQ_LENGTH, ((longest + 7) // 8) * 8)
    fields = {}
    for field, pad in (('input_ids', tokenizer.pad_token_id),
                       ('attention_mask', 0), ('labels', -100)):
        fields[field] = torch.tensor(
            [list(item[field]) + [pad] * (width - len(item[field])) for item in batch],
            dtype=torch.long,
        )
    assert (fields['labels'] != -100).any(), 'No target tokens in batch.'
    return fields

if train_token_ds is not None:
    assert (collate([train_token_ds[0]])['labels'] != -100).any()
    print('Answer-only training collator: passed.')

Tokenized training examples: 8156
Skipped due to length: 42
Answer-only training collator: passed.


## 10. Attach LoRA, train, and save adapter

Standard 4-bit backbone is frozen; only language-model adapters are trained. **Vision layers are excluded.** The adapter is saved to Google Drive. If a finished adapter exists, this stage loads it rather than unintentionally retraining. Incomplete training resumes from the latest checkpoint.

The Trainer uses an **integer** warmup-step count to avoid the `warmup_ratio` / `warmup_steps` version-compatibility issue from the prior notebook.

If CUDA runs out of memory: restart the runtime. Decrease `MAX_SEQ_LENGTH` and `MAX_CONTEXT_TOKENS` together, then run a new experiment. Any change creates a different run identity and requires both baseline and fine-tuned reevaluation.

In [ ]:
from pathlib import Path

ADAPTER_DIR = BASE_OUTPUT / 'lora_adapter'
CHECKPOINT_DIR = BASE_OUTPUT / 'checkpoints'
TRAIN_MANIFEST_PATH = BASE_OUTPUT / 'training_manifest.json'
training_manifest = None

if 'train' in RUN_STAGES or 'finetuned' in RUN_STAGES:
    gc.collect()
    torch.cuda.empty_cache()
    if ADAPTER_DIR.joinpath('adapter_config.json').exists() and TRAIN_MANIFEST_PATH.exists():
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
        training_manifest = read_json(TRAIN_MANIFEST_PATH)
        print('Loaded existing LoRA adapter:', ADAPTER_DIR)
    elif 'train' in RUN_STAGES:
        from unsloth import FastModel
        from transformers import Trainer, TrainingArguments
        from transformers.trainer_utils import get_last_checkpoint

        model = FastModel.get_peft_model(
            model,
            r=LORA_R,
            target_modules=[
                'q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj',
            ],
            finetune_vision_layers=False,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            lora_alpha=LORA_ALPHA,
            lora_dropout=0,
            bias='none',
            use_gradient_checkpointing='unsloth',
            random_state=SEED,
            use_rslora=False,
            loftq_config=None,
        )
        free_before_train, _ = torch.cuda.mem_get_info()
        print(f'Free VRAM after LoRA attachment: {free_before_train/2**30:.2f} GiB')
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f'Trainable parameters: {trainable:,} / {total:,} = {100*trainable/total:.3f}%')
        assert trainable > 0

        # Gradient checkpointing is handled by Unsloth.
        warmup_steps = max(1, round(MAX_STEPS * 0.03))
        arguments = TrainingArguments(
            output_dir=str(CHECKPOINT_DIR),
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUMULATION,
            learning_rate=LEARNING_RATE,
            max_steps=MAX_STEPS,
            warmup_steps=warmup_steps,
            lr_scheduler_type='cosine',
            weight_decay=0.05,
            fp16=True,
            bf16=False,
            optim='paged_adamw_8bit',
            logging_steps=10,
            save_strategy='steps',
            save_steps=min(SAVE_EVERY_STEPS, MAX_STEPS),
            save_total_limit=2,
            report_to='none',
            remove_unused_columns=False,
            gradient_checkpointing=False,
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            seed=SEED,
        )
        trainer = Trainer(
            model=model,
            args=arguments,
            train_dataset=train_token_ds,
            data_collator=collate,
        )
        resume_ckpt = get_last_checkpoint(str(CHECKPOINT_DIR)) if CHECKPOINT_DIR.exists() else None
        if resume_ckpt:
            print('Resuming training from:', resume_ckpt)

        train_start = time.time()
        try:
            train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
        except torch.cuda.OutOfMemoryError as exc:
            raise RuntimeError(
                'Gemma 3 12B QLoRA exceeded the T4 VRAM. Restart Colab; '
                'reduce MAX_SEQ_LENGTH and MAX_CONTEXT_TOKENS. '
                'Do not report incomplete training as a result.'
            ) from exc

        trainer.save_model(str(ADAPTER_DIR))
        tokenizer.save_pretrained(str(ADAPTER_DIR))
        from importlib.metadata import version as package_version
        package_versions = {name: package_version(name) for name in (
            'unsloth', 'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes'
        )}
        training_manifest = {
            **EXPERIMENT_CONFIG,
            'package_versions': package_versions,
            'training_id': str(uuid.uuid4()),
            'warmup_steps': warmup_steps,
            'optimizer': 'paged_adamw_8bit',
            'trainable_parameters': trainable,
            'total_parameters': total,
            'tokenized_training_rows': len(train_token_ds),
            'skipped_long_targets': skipped_long_targets,
            'train_metrics': train_result.metrics,
            'train_elapsed_seconds': round(time.time() - train_start, 2),
            'peak_allocated_gpu_gib': round(torch.cuda.max_memory_allocated() / 2**30, 3),
            'torch_version': torch.__version__,
        }
        write_json(TRAIN_MANIFEST_PATH, training_manifest)
        print('LoRA adapter saved:', ADAPTER_DIR)
        print('Training metrics:', train_result.metrics)
    else:
        raise FileNotFoundError(
            'Adapter not found. Run the training phase first, then evaluate.'
        )
else:
    print('Training and adapter loading skipped.')

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Free VRAM after LoRA attachment: 6.97 GiB
Trainable parameters: 32,735,232 / 6,635,330,160 = 0.493%


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,156 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 32,735,232 of 12,220,060,272 (0.27% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.865170
20,0.479280
30,0.473857
40,0.389624
50,0.250392
60,0.427608
70,0.543059
80,0.267935
90,0.300226
100,0.318244


LoRA adapter saved: /content/drive/MyDrive/Gemma3_12B_PersianQA_T4/full_6f853880f866/lora_adapter
Training metrics: {'train_runtime': 5077.755, 'train_samples_per_second': 0.394, 'train_steps_per_second': 0.049, 'total_flos': 5.054191852086144e+16, 'train_loss': 0.3864832210540772, 'epoch': 0.24521824423737126}


## 11. Evaluate fine-tuned 12B on the identical held-out split

All 930 (or other dataset-revision-dependent count) rows are processed. Progress messages like `900/930` are intermediate; only `930/930` and an equal-length prediction file indicate completion.

In [ ]:
finetuned_rows = None
finetuned_scores = None
if 'finetuned' in RUN_STAGES:
    finetuned_rows, finetuned_scores = evaluate_model(model, 'finetuned', official_test)
    print(json.dumps(finetuned_scores, ensure_ascii=False, indent=2))
else:
    print('Fine-tuned evaluation skipped.')

## 12. Paired comparison and research report

Reloads predictions from Drive when necessary, checks that all references and IDs align, and computes paired bootstrap 95% confidence intervals over examples. Reports overall F1, answerable F1, Exact Match, and abstention F1. No result is fabricated if one stage is incomplete.

In [ ]:
def paired_bootstrap(base, tuned, field='f1', n_bootstrap=BOOTSTRAP_REPEATS):
    before = np.array([x[field] for x in base], dtype=np.float64)
    after = np.array([x[field] for x in tuned], dtype=np.float64)
    assert before.shape == after.shape and len(before) > 1
    deltas = after - before
    generator = np.random.default_rng(SEED)
    draws = np.empty(n_bootstrap, dtype=np.float64)
    for i in range(n_bootstrap):
        positions = generator.integers(0, len(deltas), size=len(deltas))
        draws[i] = deltas[positions].mean()
    low, high = np.quantile(draws, [0.025, 0.975])
    return {'delta': float(deltas.mean()), 'ci95': [float(low), float(high)]}

if 'compare' in RUN_STAGES:
    base = read_saved_records('baseline')
    tuned = read_saved_records('finetuned')
    assert len(base) == len(tuned) == len(official_test), 'Incomplete baseline or tuned evaluation.'
    validate_saved_records(base, official_test)
    validate_saved_records(tuned, official_test)
    assert [x['id'] for x in base] == [x['id'] for x in tuned]
    assert [x['references'] for x in base] == [x['references'] for x in tuned]
    assert (BASE_OUTPUT / 'training_manifest.json').exists(), 'Training manifest not found.'
    for name in ('baseline', 'finetuned'):
        assert (BASE_OUTPUT / f'{name}.metrics.json').exists(), f'Missing completed {name} metrics.'
        assert read_json(BASE_OUTPUT / f'{name}.evaluation_manifest.json') == evaluation_identity(name)

    baseline_scores = read_json(BASE_OUTPUT / 'baseline.metrics.json')
    finetuned_scores = read_json(BASE_OUTPUT / 'finetuned.metrics.json')
    f1_comparison = paired_bootstrap(base, tuned, 'f1')
    em_comparison = paired_bootstrap(base, tuned, 'em')
    answerable = [(x, y) for x,y in zip(base, tuned) if not x['gold_no_answer']]
    answerable_f1_comparison = paired_bootstrap(
        [x for x,y in answerable], [y for x,y in answerable], 'f1'
    )

    comparison = {
        **EXPERIMENT_CONFIG,
        'n': len(official_test),
        'baseline': baseline_scores,
        'fine_tuned': finetuned_scores,
        'paired_f1': f1_comparison,
        'paired_em': em_comparison,
        'paired_answerable_f1': answerable_f1_comparison,
        'training_manifest': str(TRAIN_MANIFEST_PATH),
        'status': 'SMOKE TEST — not reportable' if SMOKE_TEST else 'Full held-out evaluation',
    }
    write_json(BASE_OUTPUT / 'comparison.json', comparison)

    examples = [
        {
            'id': x['id'], 'question': x['question'], 'references': x['references'],
            'baseline_prediction': x['prediction'],
            'finetuned_prediction': y['prediction'],
            'baseline_f1': x['f1'], 'finetuned_f1': y['f1'],
            'delta_f1': y['f1'] - x['f1'],
            'gold_no_answer': x['gold_no_answer'],
        }
        for x, y in zip(base, tuned)
    ]
    with (BASE_OUTPUT / 'per_example_comparison.jsonl').open('w', encoding='utf-8') as writer:
        for item in examples:
            writer.write(json.dumps(item, ensure_ascii=False) + '\n')

    def pct(number): return f'{number * 100:.2f}%'
    def pp(number): return f'{number * 100:+.2f}'

    report = f'''# Gemma 3 12B PersianQA controlled experiment

- **Status:** {'SMOKE TEST — NOT RESEARCH EVIDENCE' if SMOKE_TEST else 'Full held-out evaluation'}
- Model: `{MODEL_ID}` (`{MODEL_SHA}`)
- Dataset: `{DATASET_ID}`; converted Parquet revision: `{PARQUET_SHA}`
- Official held-out split: `{OFFICIAL_TEST_SPLIT}` (N={len(official_test)})
- LoRA: rank={LORA_R}, alpha={LORA_ALPHA}, steps={MAX_STEPS}, lr={LEARNING_RATE}
- Decoding: greedy, max_new_tokens={MAX_NEW_TOKENS}, max_context_tokens={MAX_CONTEXT_TOKENS}

| Metric | Original 12B | Fine-tuned 12B | Difference (pp) |
|---|---:|---:|---:|
| Token F1 | {pct(baseline_scores['f1'])} | {pct(finetuned_scores['f1'])} | {pp(f1_comparison['delta'])} |
| Exact Match | {pct(baseline_scores['em'])} | {pct(finetuned_scores['em'])} | {pp(em_comparison['delta'])} |
| Answerable-question F1 | {pct(baseline_scores['answerable_f1'])} | {pct(finetuned_scores['answerable_f1'])} | {pp(answerable_f1_comparison['delta'])} |
| No-answer F1 | {pct(baseline_scores['no_answer_f1'])} | {pct(finetuned_scores['no_answer_f1'])} | {pp(finetuned_scores['no_answer_f1'] - baseline_scores['no_answer_f1'])} |

Paired bootstrap 95% CI of F1 difference (pp): [{pp(f1_comparison['ci95'][0])}, {pp(f1_comparison['ci95'][1])}].
Paired bootstrap 95% CI of EM difference (pp): [{pp(em_comparison['ci95'][0])}, {pp(em_comparison['ci95'][1])}].
Paired bootstrap 95% CI of answerable F1 difference (pp): [{pp(answerable_f1_comparison['ci95'][0])}, {pp(answerable_f1_comparison['ci95'][1])}].

## Limitations
- A 12B T4 configuration has a smaller context budget than the previous 4B run; do not directly attribute differences to model scale alone.
- Contexts shortened by token budget: {baseline_scores['truncated_contexts']}/{len(official_test)}.
- Answerable truncated examples missing every gold span in the remaining context: {baseline_scores['answerable_gold_missing_after_truncation']}.
- This is a single-seed, {MAX_STEPS}-step adaptation, not a hyperparameter sweep or proof of convergence.
- Exact abstention phrase matching can miss equivalent Persian abstention wording.
- Normalized EM and token F1 may differ from the dataset authors' official implementation.
'''
    (BASE_OUTPUT / 'report.md').write_text(report, encoding='utf-8')
    print(report)
    print('Report saved:', BASE_OUTPUT / 'report.md')
else:
    print('Comparison stage skipped.')

## 13. Output files and interpretation

The output directory contains `experiment_config.json`, `baseline.predictions.jsonl`, `baseline.metrics.json`, `checkpoints/`, `lora_adapter/`, `training_manifest.json`, `finetuned.predictions.jsonl`, `finetuned.metrics.json`, `per_example_comparison.jsonl`, `comparison.json`, and `report.md` once all stages complete.

**Never present synthetic examples, partial predictions, or a smoke-test score as a completed experiment.** Do not reuse the 4B result (F1 81.24%) as a 12B result. The 12B baseline and fine-tuning results remain unknown until this notebook executes successfully.

**Resume:** Reconnect Drive, select the needed stage(s) in `RUN_STAGES`, and rerun setup cells. If training was interrupted, checkpoints under `checkpoints/` are used. If evaluation was interrupted, saved JSONL predictions are reused. Changing the model, LoRA rank, sequence budget, dataset revision, or training steps creates a new output directory.

**VRAM caveat:** Standard 4-bit weights are substantially smaller than Unsloth's dynamic checkpoint, but QLoRA on a single T4 may still fail. If that occurs, no successful-run claim should be made. Smaller training context, fewer adapted projection types or CPU offloading alter the experiment and must be documented.